# Smooth and gap-free NDVI of Swiss forests

Authors: Fabian Bernhard, Francesco Grossi, Benjamin D. Stocker

Version: 1.0 - 26.05.2026


<a name = "ToC"></a>

## Table of Contents
_[TODO: To be regenerated manually]_

<a id="T1"></a>
## 1. Aim
Vegetation health and activity can be monitored based on remotely sensed data of the Normalized Difference Vegetation Index (NDVI). Sentinel-2 provides NDVI data at high spatial resolution (10 m) at frequent revisit times (up to 5 days) and an additionally enhanced data product (swissEO S2-SR), covering the area of Switzerland, is made available through swisstopo. However, challenges for data interpretability and usability for operational vegetation health (https://www.swisstopo.admin.ch/de/satellitenbilder-swisseo-vhi) and national drought monitoring (https://www.trockenheit.admin.ch/en) remain. Challenges arise from data gaps, the large scatter, and apparent outliers in the data. The data availability largely depends on the overpass date of the Sentinel-2 satellites, cloud, and snow cover (Fig. 1).

<figure style="text-align: center;">
    <img src="illustrations/00_NDVI_availability.png" style="width:30%;">
    <figcaption>Figure 1: Illustration of data availability of NDVI values during October 2025 in the swissEO S2-SR product (screenshot from https://www.swisstopo.admin.ch/en/satelliteimage-swisseo-s2-sr).</figcaption>
</figure>

In this project, we create a workflow to process the NDVI data from swissEO S2-SR, creating a smooth and gap-free data product at the daily scale and 10 m resolution, covering forest areas Switzerland. The processing includes:

- Filtering of observations based on cloud and snow cover data
- Identification of outlier observations
- Smoothing and interpolation of the observations

The workflow is applied to available data from April 2017 to November 2025 (historical processing) and a setup to work with newly incoming data is developed and presented. Accordingly, the project yields two products:

1. **Historical processing (HP)**: A smooth and gap-free daily NDVI product, based on S2-SR, covering past dates (April 2017 to November 2025). These data are made available as Cloud Optimized GeoTIFF files.
2. **Continuous integration (CI)**: An algorithm to continuously ingest incoming NDVI data from S2-SR and update cleaning, gapfilling, and smoothing in consistency with the _Historical processing_. 



<a id="T2"></a>
## 2. Methodological approach

The approach taken is based on a transformation of NDVI observations into anomalies with respect to a site-specific (pixel-specific) mean seasonal cycle (MSC) of the NDVI (Sec. 2.1). Anomalies are then processed:

- Data cleaning (Sec. 2.2)
- Outlier detection (Sec. 2.3)
- Smoothing and interpolation (Sec. 2.4)

... and added back to the MSC for the final smooth and gap-free product. 

Largely the same method is applied for the historical processing and continuous integration, but with adjustments to enable practicability of the implementation of the latter. The implementation is described in Sec. 3.

NDVI was derived from spectral reflectances ($\rho$), obtained from the S2-SR data, as 
$$\mathrm{NDVI} = \frac{\rho_{\mathrm{NIR}} - \rho_{\mathrm{RED}}}{\rho_{\mathrm{NIR}} + \rho_{\mathrm{RED}}},$$
where $\rho_{\mathrm{NIR}}$, $\rho_{\mathrm{RED}}$, $\rho_{\mathrm{GREEN}}$, $\rho_{\mathrm{SWIR}}$ (reprojected from 20 m to 10 m resolution) are the reflectances of the corresponding bands.

All steps were applied to pixels, classified as *forest*. Forest area is defined by the _swissEO S2-SR_ product (https://www.swisstopo.admin.ch/de/satellitenbilder-swisseo-vhi) from 2025-05-01 where the vegetation health index is not 255.

<!-- Figure 2 illustrates the approach, whereby outliers (red dots) are identified and removed, and observations (blue dots) and unobserved daily values (empty dots) are smoothed using a LOESS spline (hatched dots spline not shown).

<figure style="text-align: center;">
    <img src="illustrations/01c_NDVI_residual.png" style="width:30%;">
    <figcaption>Figure 2: Illustration of a time series of NDVI values (dots) for a single pixel and the modelled median NDVI (solid line). Observations are shown as colored dots (in blue 'observations', in red 'outliers'), `L2` output (i.e. smoothed and gapfilled) is shown as hatched dots. `delta` distance is indicated by arrow.</figcaption>
</figure> -->

<!-- XXX TBC: Make this a nice figure XXX -->


<a id="T2.1"></a>
### 2.1 Mean seasonal cycle

All NDVI data was first transformed to anomalies with respect to the mean seasonal cycle:
$$\Delta x(i,d,y) = x(i,d,y) - \overline{x}(i,d),$$
where $\Delta x$ are the anomalies, $i$ is the location (pixel), $d$ is the day-of-year, $y$ is the year, and $\overline{x}(d)$ is the average NDVI for a given day-of-year, i.e., the mean seasonal cycle. $\overline{x}(d)$ was defined based on <a href="#ref1">Biegel et al., 2026</a> (see Sec. 2.1) and taken as 
$$
\overline{x}(i,d) = (q_{75}(i,d) - q_{25}(i,d)) / 2,
$$
where $q_{75}$ and $q_{75}$ are the modelled quantiles, taken from <a href="#ref1">Biegel et al., 2026</a>. 

In summary, the modelling of the NDVI MSC quantiles was done by <a href="#ref1">Biegel et al., 2026</a> as follows. A double sigmoid function is assumed to represent the annual course of NDVI, defined by six parameters (minimum, maximum, start of season, end of season, duration of "green-up", duration of "brown-down"). A single model (multilayer perceptron) is trained to predict these six parameters for each location (gridcell), given a set of time-invariant features. Each quartile of the NDVI, given the day-of-year (DOY) and location is predicted. Time-invariant features are the median forest height, forest mix rate, dominant tree species, habitat type, and various topographical indices, derived from the SwissAlti3D digital elevation model (sources given in <a href="#ref1">Biegel et al., 2026</a>).

In Biegel et al. (2026), the NDVI MSC is used to identify anomalies, while no smoothing and gapfilling was performed. Here, we use the NDVI MSC as a basis for identifying outliers and for smoothing and gapfilling.

<a id="T2.2"></a>
### 2.2 Data cleaning

Data was first cleaned to remove snow and cloud-affected values. The identification of snow-affected values was based on the Normalised Difference Snow Index (NDSI), defined as
$$\mathrm{NDSI} = \frac{\rho_{\mathrm{SWIR}} - \rho_{\mathrm{GREEN}}}{\rho_{\mathrm{SWIR}} + \rho_{\mathrm{GREEN}}},$$
where $\rho_{\mathrm{NIR}}$, $\rho_{\mathrm{RED}}$, $\rho_{\mathrm{GREEN}}$, $\rho_{\mathrm{SWIR}}$ (reprojected from 20 m to 10 m resolution) are the reflectances of the corresponding bands. The NDSI was calculated from S2-SR data of the individual $\rho$ bands. We chose a threshold of 0.43 and discarded all NDVI data if the NDSI was above this value on the corresponding date. 

The identification of cloud-affected or missing values was based on the cloud and terrain mask, provided through S2-SR (asset names ending in `masks-10m.tif`). We discarded all NDVI data on the corresponding date, if either the cloud mask indicated `1` or `255`, or if the the terrain mask indicated `100` or `255`, or if the red or near-infrared channel indicated `9999`.

<a id="T2.3"></a>
### 2.3 Outlier detection

A data point $x(i,d,y)$ was considered an *outlier* if it fulfilled both of the following two criteria:

- The deviation from the MSC exceeded a threshold of 0.05. That is, if $\Delta x(i,d,y) > 0.05$.
- The deviation from the neighbouring values exceeded a threshold of 0.1. That is, if $|\Delta \Delta x(i,d,y)| > 0.01$, where
$$
\Delta \Delta x(i,d,y) = \Delta x(i,d-1,y) - \Delta x(i,d,y).
$$
This considers data of the previous available date. This criterium was anaologously evaluated also for the subsequent available date.

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">

<i>Improvement:</i>
Alternatively, one could consider time-normalised $\Delta \Delta$, that is a division by the number of days two subsequent observations lie apart. Moreover, instead of using a fixed value, this threshold could be derived based on statistical distribution of values throughout (2018-2025 (remove 2017 because bigger differences in time between acquisitions)) – either pixel-specific or whole-Switzerland statistics.

</div>

The choice of fixed global thresholds was based findings from on alternative tests. Tests (not shown) were done to explore a pixel and date-wise IQR-based definition of the $\Delta x$ threshold, and a pixel-wise quantile-based definition of the $\Delta \Delta x$ threshold. These alternatives definitions of thresholds yielded ineffective outlier removal for a large fraction of pixels and dates.

<a id="T2.4"></a>
### 2.4 Smoothing and interpolation

A LOESS spline was applied to data points $\Delta x(i,d,y)$, retained after data cleaning and outlier removal, described Secs. 2.2 and 2.3. The LOESS spline assumed equidistant dates and used a `frac`-parameter of 1 (i.e. the fraction of the data used when estimating each y-value) and three iterations of residual-based reweightings wer performed (parameter `it`) with the implementation in `statsmodels.api.nonparametric.lowess()`. Only the LOESS-smoothed values of dates for which original observations were available from S2-SR and not removed were retained. Values for dates in between were linearly interpolated from the smoothed values. 

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">

<i>Improvement:</i>
Values for dates in between the observations were linearly interpolated. As an alternative one could use the smoothed value from LOESS spline at each date and thus avoid an additional step of linear interpolation for the `L2` and `L1` points. Only `L0` extrapolation would be treated separately. This would require to switch to the use of _non-_ equidistant dates. Note that even when using the LOESS spline for interpolation, the distinction between observation and non-observation dates would still be needed in order to only use observation dates for LOESS splines of subsequent updates.

</div>

<!-- _[TODO: see what was previously written in section "Test different approaches" in: https://github.com/geco-bern/swiss-ndvi-processing/blob/27d1061a31d313d6fe428466d4b7771e0e18ccfd/report/method_report.ipynb]_ -->

<!-- #### 2.1.1. Definitions


- `CI`: designates the continuous ingestion pipeline, as opposed to the historical processing.
- `L2`, `L1`, `L0`: designate different stages of output values from the continuous ingestion pipeline (see Figure X for illustration). `L0` is between the last observation and today, `L1` temporary values within the smoothing window (still subject to change upon reruns), and `L2` values are fixed and will not be overwritten on reruns. (Internally these are encoded as integer value as specified in Table 3. _[`TODO: update this and use L level as tens: e.g. use levels 1,2,3,4 for L0, levels 11,12,13,14 for L1 and levels 21,22,23,24 for L2.`]_) -->


## 3. Implementation

Steps for cleaning, outlier removal, and smoothing (described in Sec. 2) were applied per pixel. This enables parallelisation of the computations (see Sec. XXX: refer to section where computation requirements and performance are described XXX). Implementations for the historical processing (HP) and for the continuous integration (CI) required somewhat different implementations, but were based on largely identical methodological steps, yielding mutually consistent outputs (Sec. XXX: refer to section where outputs, files, are described and where some test of this consistency is presented, if not to be done XXX).

### 3.1 Historical processing (HP)

For HP, all available S2-SR data is downloaded. Then, NDVI and NDSI are computed from spectral reflectances (see above). Finally, a function implementing all processing steps (cleaning, outlier removal, smoothing, see Sec. 2) is applied once per pixel and for the full time series (2017-04-01 to 2025-12-31). 

For days with multiple satellite overpasses (S2A and S2B), generating multiple observations for a given date, only data from the first overpass is used. Here, this approach discarded a total of 30 acquisitions.

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
    <i>Improvement:</i>
    These 30 acquisition could be kept with an alternative approach that considers the average value of multiple overpasses.
</div>

The following scripts and functions implement the steps described above:

- `xxx`: downloading S2-SR data
- `xxx`: computing NDVI and NDSI
- `xxx`: cleaning, outlier removal, smoothing
- ...

The following files are generated:

- `xxx` (TBC: explain file name, format, describe variables: units etc., approx. file size; also TBD: made publicly available?)

### 3.2 Continous integration (CI)

For the continuous ingestion (CI), the algorithm is repeatedly applied on a moving window, thereby defining output values of different preparation stages: `L0`, `L1`, `L2`. Figure 3 shows the application of the smoothing and gapfilling procedure.

XXX TBC: this section needs revision to better explain the approach taken. The following aspects should be clearly explained: moving window width, iteration (moving window shift) each day or only when new observation comes in?, approach taken to expand beyond last observation, latency of last L2 observation, which dates may be overwritten again and how this is indicated by L...

<figure style="text-align: center;">
    <img src="illustrations/swisstopo_continuous_updated_01.png" style="width:100%;">
    <figcaption>Figure 3: Illustration of continuous integration. Green and blue dots indicate raw and smoothed observations, respectively. Black dots and empty dots with dashed contour indicate days without observations. The green box “7-observation moving window” shows the set of observations used for LOESS smoothing, and the dashed red boxes show the updated values (n=3 in this example). [The larger dot-dashed box shows the set of daily values considered (and re-written) at each continuous processing, of which most values remain unchanged.]</figcaption>
</figure>

<figure style="text-align: center;">
    <img src="fig/historical_2026-04-04_18h16_historical_v7c_CI-updated.zarr-update-2026-05-25_20h29.png" style="width:100%;">
    <figcaption>Figure 4: Illustration of evolution of pixel processing status with date. Left panel uses the `mask_array` code, whereas right panel aggregates those to L2/L1 processing status. [TODO: update with latest CI run. And include L0.] </figcaption>
</figure>

XXX: can text in the next three paragraphs below be integrated into caption of Fig 3 or Fig 4? 

The procedures of 1) outlier detection and 2) LOESS smoothing and 3) extrapolation are indicated by the vertical arrows. Values that are still subject to change and are labelled `L0` or `L1`. Values outside labelled `L2` are not modified anymore. In Figure 3 the five values that are updated are marked with dashed red boxes in the lower panel. Two values have newly reached status `L2`; one value has reached status `L1`; and two values are extrapolated `L0`.

Note: The implementation makes use of a larger window (indicated by the dot-dashed box), which for simplicity of the implementation is entirely re-written at each application. Since only the values in the dashed red boxes are updated, most of the re-written values remain unchanged.

The moving window width is seven observations. Note that this definition implies that the window width in number of days is variable, depending on the number of non-observation days. Given that the window width depends on the observation record, it is thus pixel-specific.

Outlier detection requires a preceeding and following observation. In the continuous ingestion (CI), the outlier detection can thus still change when new data becomes available. Some outputs values are thus still marked as tentative (`L1`), and fixed values are only attributed to observation positions from the earlier half of the moving window.

<!-- Additional text:

The smoothing windows covers all the observations in the historical setup and has a fixed dimensions of 7 elements in the continous integration. We decided an odd number of element so that there is a single data that is perfectly centered.

We tried different smoothing windows, with 5, 7 and 9 elements.

With 5 elements, the smoothing does not appear to be effective beacuse there is a lot of variability compared to a smoothing using a window of 7 elements.

The differences between the smoothing with a window element of 7 and 9 are negligible. The minimal gain in smoothing is ... by the increase in latency,

We decided to keep the constant smoothing window with 7 elements to minimize the latency while providing an effective smoothing. -->

<!-- ### 3.3 Smoothing window
_[TODO: see what was previously written in section "Test different approaches" in: https://github.com/geco-bern/swiss-ndvi-processing/blob/27d1061a31d313d6fe428466d4b7771e0e18ccfd/report/method_report.ipynb]_ -->


<!-- <a id="T2.2"></a>
### 2.2 Historical and continuous processing (CI) -->

<!-- Repeated application of the window in the CI processing is illustrated below by Figure 4. It shows again the first step from Figure 3 (Fig. 4a) and illustrates the subsequent updates on a day after and additional acquisition and a non-observation day (Fig. 4b) and after another additional acquisition (Fig. 4c).

<figure style="text-align: center;">
    <img src="illustrations/swisstopo_continuous_updated_02.png" style="width:50%;">
    <figcaption>Figure 4: Repeated application of the moving window for the continuous processing. The repeated steps are shown in panels a), b), c), respectively.</figcaption>
</figure> -->

<style>
table {
    margin-left: 0 !important;
    margin-right: auto !important;
}
</style>


## 4. Results

We tested the historical processing on eight example sites, representing different climatic conditions, forest types

For each case, we show the historical processing and an area visualization comparison between the raw and processed NDVI. We selected to show the areal view of 2021-08-12 and 2023-07-20 because these two dates correspond to one date before and one date shortly after Bitsch forest fire (July 17th 2023), respectively. The area is a square of 10 km centered around the coordinates listed below for most sites (except for the drougth and storm-affected sites, where the maps show a 1.2 km square).

Table 1: Example sites sites used for evaluation and their coordinates
| Example site                          | Coordinates (CH1903-LV95)       |     
|---------------------------------------|---------------------|
| Lowland broadleaf                     | 2694491, 1126023    |
| Highland broadleaf                    | 2692020, 1121443    |
| Lowland evergreen                     | 2761097, 1194613    |
| Highland evergreen                    | 2781537, 1182974    |
| Bitsch fire affected area             | 2644029, 1134128    |
| Bitsch fire nearby non-affected area  | 2644328, 1134342    |
| Drought-affected area (2018, Schaffhausen)      | 2690025, 1287413    |
| Storm-affected area (2021, Airolo)    | 2689564, 1154411 |

Aerial images of the example sites are provided as links to an online map in Appendix A.

<!-- (XXX:display site location on a Switzerland map) -->

<!-- Illustration of continuous ingestion (TODO: not updated):
We mimic the continuous ingestion by simulating the data from February 2022 to November 2022 by processing one date at time in the continuous setup and show the final results as GIF. We show two of the aforementioned cases, the Lowland broadleaf and Highland evergreen. 

We choose to show these two location due to the frequency of incoming NDVI data; in the first case the raw data are of good quality and the latency between the current and smoothed date is low, in the latter most of the incoming data are invalid (mostly due to snow), so in this case there will be a significant lag between the current and smoothed date. In this case multiple invalid data are ingested in sequence. In this case the workflow cannot proceed due to the lack of data themselves. -->


<a id="T4.1"></a>
### 4.1 Time series

_Results are generated with `3_check_historical_ndvi.py`._

<figure style="text-align: center;">
    <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig0_evaluationSites.png" style="width:100%;">
    <figcaption>Figure 5: Time series resulting from historical processing for the focus sites. Raw observations are shown as green dots, smoothed observations as black (non-outlier) or red crosses (outlier) and smoothed and gapfilled NDVI as solid black line. The expected seasonal evolution of NDVI ('median') is overlaid as solid, grey line.</figcaption>
</figure>

<!-- XXX would be nice: mark raw data (circles) in red, not interpolated (crosses). XXX -->
<!-- XXX would be nice: mark dates of known events -->

#### General observations across all sites

Figure 5 shows the time series resulting from the historical processing at the focus sites (single pixels extracted). The top four rows show forests without major perturbations representing broadleaf and evergreen forests in both lowlands and highlands (alpine regions). Raw observations are shown as green dots, smoothed observations as black or red crosses (depending on their outlier status) and smoothed and gapfilled NDVI as solid black line. Orange crosses depict observations among the last four within the moving window. These observations are still to smooth. <!-- TODO after post-processing these should be removed. Better speak in terms of L2, L1, L0. -->
The expected seasonal evolution of NDVI ('median') is overlaid as solid, grey line.

The comparison between sites illustrates the differences in seasonal patterns between different forest types and elevations.

The sites with perturbations (storm (late 2018) and Bitsch fire (17.07.2023)) showcase how the algorithm is able to correctly switch to a new normal seasonality in spite of large differences with the `median`. The algorithm succeeds in refraining from flagging new extreme values as outliers since neighboring observations are confirming the "new normal".

Generally, dynamics are determined by the median even during periods with few observations (see evolution of low NDVI during winter 2017/2018, e.g. the lowland broadleaf site or any other). In spite of the linear interpolation between observations, the gapfilled data can reproduce smooth dynamics following the sigmoid shape because gapfilling is performed in `delta` space followed by addition of the sigmoid-shaped `median`.

Some deviations from a smooth interpolated signal can be spotted at the Bitsch nearby site in winters 2020/2021 and 2023/2024 as well as at the highland evergreen site in winter 2020/2021. Note how in these cases two subsequent observations indicated very low NDVI, were not flagged as outlier and were thus able to pull down the LOESS spline (repeatedly at the Bitsch site). 

#### Broadleaf sites

These sites show very few outliers and a rather regular trajectory during summer season, with strongest deviations from the `median` during the winter season.

#### Evergreen sites

The lowland site shows very little seasonal variation in the `median` and in the raw observations. Outliers appear to cluster during winter season. The highland site shows slightly stronger seasonal variations, with strong drops in the 2023/2024 and 2020/2021 winters. The beginning of 2020/2021 winter showed in both evergreen sites a consistent spike in NDVI (also visible in the Bitsch non-affected area).
There might be a tendency for larger deviations of raw observations from `median` in highlands versus lowlands. There also appears to be a tendency for larger deviations during winter season than summer season.

#### Perturbed sites

The sites with perturbations (storm (actually 2018, but drop in late 2020 [1]) and Bitsch fire (17.07.2023)) did show the change to new normal. The drought affected area did show a drop in observed values in late summer. The smoothed and gapfilled data also shows this decrease but slightly less pronounced than the observations due to the smoothing. Thus severity and duration of the NDVI decrease will be slightly underestimated based on the smoothed and gapfilled data.


Suprisingly, high NDVI observations in early winter 2017/2018 (i.e. before the fire) were gapfilled differently between the two Bitsch sites. While in the (later) fire-affected site the high observation values were flagged as outliers, in the the nearby non-affected site the high observation values were used for interpolation and appear to have influenced the values of summer 2017. The algorithm does not control the maximum width of the 7-observation-window in days, which can lead to these long-ranging influences. In a future variant this might be capped. However, it appears that this issue is not wide-spread. Observation density of the data after 2017 had considerably increased and it appears. There appears to be only one occurrence after 2017 of such a long-ranging influence contradicting the observations among our focus sites, which appears to have been in the highland evergreen site during winter 2023/2024 causing a drop in late summer values.

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">

<i>Improvement:</i>
Note that the above analysis reflects the patterns shown by individual pixels of the generated maps. If these maps are used for spatial aggregation, an future evaluation could involve plotting the average and spread of gapfilled NDVI values from 25 pixels around the individual pixel to assess the persistence of the observed patterns in space (compared to the dynamic from an individual pixel).

</div>

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">

[1] Note that according to the aerial image (https://s.geo.admin.ch/veber3scah7o) the coordinates (2689564,1154411) of the storm-influenced site north of Airolo appear to be centered in a large wind-throw area. This was attributed to the Vaia strom (29/30 October 2018). Interestingly, the temporally consistent drop in NDVI appears to happen only in late 2020. This might be linked to imprecise centering of the coordinates (2689564,1154411), subsequent cleanup works or a subsequent storm. For the Vaia storm, nearby coordinates (2689741,1154618) could show a more pronounced decrease in NDVI in October 2018 since they were also identified by a decrease in vegetation height in 2019 (https://www.swisstopo.admin.ch/en/drought-media-collection#Storm-Vaia-above-Airolo--True-colour-images-and-VHI-map-(2018)).

</div>



<a id="T4.2"></a>
### 4.2 Maps
_Results are generated with `3_check_historical_ndvi.py`._

Figure 5 shows maps for two dates for the two broadleaf sites, and Figure 6 shows the same for the evergreen sites. Grey pixels indicate forested area with missing data, that is successfully gapfilled by in the processed output. Observations were missing due to cloudiness, resulting in grey patches (Fig 5), or due to non-acquisition date, resulting in whole area being grey (Fig 6).

Figure 7 shows additionally the processing and outlier status encoded in `mask_array` to assess the outlier patterns.

Figure 8 show maps for two dates for the three perturbation sites.

#### General observations across all sites
For the cloudy pixels: On the selected dates, there is no apparent artifact by gapfilling the cloudy pixels when judging by the agreement with surrounding non-cloudy pixels. 

Smoothing and outlier removal does inherently reduce variance. A strong smoothing of signal visible in the mountainous area in the south east part of Figure 6b. 
Smoothed outliers are marked with 4, gapfilled missing observations with 2, and smoothed observations with 3.

<style>
  .two-col-combined-fig {
    display: grid;
    grid-template-columns: repeat(2, 1fr);
    gap: 1rem;
    align-items: start;
    justify-items: stretch;
  }
  .panel {
    position: relative;
    width: 100%;
    background: #fff;
  }
  .panel img {
    width: 100%;
    height: auto;
    display: block;
    border: 1px solid #ddd;
    box-shadow: 0 1px 2px rgba(0, 0, 0, 0.05);
  }
  .panel .panel-label {
    position: absolute;
    left: 8px;
    top: 8px;
    background: rgba(255, 255, 255, 0.85);
    padding: 6px 8px;
    font-weight: 700;
    border-radius: 4px;
    font-size: 0.95rem;
    box-shadow: 0 1px 2px rgba(0, 0, 0, 0.08);
  }
  .combined-caption {
    grid-column: 1 / -1;
    text-align: center;
    font-size: 0.95rem;
    color: #222;
    margin-top: 6px;
    padding-top: 8px;
    border-top: 1px solid #eee;
  }
  @media (max-width: 900px) {
    .two-col-combined-fig {
      grid-template-columns: 1fr;
    }
  }
</style>

#### Broadleaf sites

<div class="two-col-combined-fig" style="width:70%;">
    <figure class="panel">
        <span class="panel-label">a)</span>
        <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig2_evaluationSites__low_blf.png" alt = "__low_blf";">
        <!-- <figcaption>Figure 5: Maps of processed NDVI for two dates for the focus site: lowland broadleaf. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.</figcaption> -->
    </figure>
    <figure class="panel">
        <span class="panel-label">b)</span>
        <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig2_evaluationSites__high_blf.png" alt = "_high_blf";">
        <!-- <figcaption>Figure 6: Maps of processed NDVI for two dates for the focus site: highland broadleaf. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.</figcaption> -->
    </figure>
    <div class="combined-caption">
    Figure 5: Maps of processed NDVI for two dates for the broadleaf focus sites: a) lowland, b) highland. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.
    </div>
</div>


#### Evergreen sites

<div class="two-col-combined-fig" style="width:70%;">
    <figure class="panel">
        <span class="panel-label">a)</span>
        <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig2_evaluationSites__low_enf.png" alt = "__low_enf";">
        <!-- <figcaption>Figure 7: Maps of processed NDVI for two dates for the focus site: lowland evergreen. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.</figcaption> -->
    </figure>
    <figure class="panel">
        <span class="panel-label">b)</span>
        <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig2_evaluationSites__high_enf.png" alt = "_high_enf";">
        <!-- <figcaption>Figure 8: Maps of processed NDVI for two dates for the focus site: highland evergreen. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.</figcaption> -->
    </figure>
    <div class="combined-caption">
    Figure 6: Maps of processed NDVI for two dates for the evergreen focus sites: a) lowland, b) highland. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.
    </div>
</div>

<figure style="text-align: center; width:35%;">
    <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig3_evaluationSites__low_enf.png" >
    <figcaption>Figure 7: Maps of processed NDVI for two dates for the lowland evergreen site. Top row shows the processed mask label of each pixel (see Table 2), center row shows processed data for all forest pixels, bottom row shows observations, missing observations are shown in grey.</figcaption>
</figure>


#### Perturbed sites

The Bitsch forest fire (Fig 8a), lead to a strong signal three days after it happened. The processing lead to an attenuated (slower) decrease in NDVI values for this fast step change which is visible in the raw observations. 
The drought affected area (Fig 8b) shows widespread decrease in NDVI in late August 2018, which is considerably stronger than in late August 2019. In 2019 the were some patches with lower NDVI, but on much smaller scale. 
The wind-affected area (Fig 8c) clearly shows the different patches north of Airolo, which are also visible in aerial images. The two selected dates indicate that in 2019, there appeared to be some patches with lower NDVI, but the large larger windthrow-related patches only appear at later dates. Lastly, note the dark red patch with very clear contours in the south, closest to Airolo, appears to be rather due to a construction site during 2021 than to a windthrow event.



<style>
  .three-col-combined-fig {
    display: grid;
    grid-template-columns: repeat(3, 1fr); /* 3 columns */
    gap: 1rem;
    align-items: start;
    justify-items: stretch;
  }
  .panel {
    position: relative;
    width: 100%;
    background: #fff;
  }
  .panel img {
    width: 100%;
    height: auto;
    display: block;
    border: 1px solid #ddd;
    box-shadow: 0 1px 2px rgba(0, 0, 0, 0.05);
  }
  .panel .panel-label {
    position: absolute;
    left: 8px;
    top: 8px;
    background: rgba(255, 255, 255, 0.85);
    padding: 6px 8px;
    font-weight: 700;
    border-radius: 4px;
    font-size: 0.95rem;
    box-shadow: 0 1px 2px rgba(0, 0, 0, 0.08);
  }
  .combined-caption {
    grid-column: 1 / -1;
    text-align: center;
    font-size: 0.95rem;
    color: #222;
    margin-top: 6px;
    padding-top: 8px;
    border-top: 1px solid #eee;
  }
  @media (max-width: 900px) {
    .three-col-combined-fig {
      grid-template-columns: 1fr;
    }
  }
</style>


<div class="three-col-combined-fig" style="width:95%;">
  <figure class="panel">
      <span class="panel-label">a)</span>
      <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig2_evaluationSites__bitsch_fire.png" alt = "map_Bitsch_fire";">
      <!-- <figcaption>Figure 7: Maps of historical processed NDVI for two dates for the focus site: Bitsch fire-affected. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.</figcaption> -->
  </figure>
  <figure class="panel">
      <span class="panel-label">b)</span>
      <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig2_evaluationSites__drought2018.png" alt = "map_drought";">
      <!-- <figcaption>Figure 8: Maps of historical processed NDVI for two dates for the focus site: 2018 drought-affected. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.</figcaption> -->
  </figure>
  <figure class="panel">
      <span class="panel-label">c)</span>
      <img src="fig/historical_2026-04-04_18h16_historical_v7c.zarr-TESTSUITE_Fig2_evaluationSites__storm.png" alt = "map_storm";">
      <!-- <figcaption>Figure 9: Maps of historical processed NDVI for two dates for the focus site: storm-affected. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey.</figcaption> -->
  </figure>
  <div class="combined-caption">
    Figure 8: Maps of historical processed NDVI for two dates for the pertubation focus sites: a) Bitsch fire, b) drought, c) storm. Top rows show processed data for all forest pixels, bottom rows show observations, missing observations are shown in grey. (Note the different zome levels across the three subfigures.)
  </div>
</div>

<!-- <a id="T4.3"></a>
### 4.3 Animation of continuous update
_[TODO: copy paste results from sections 12.9 from `report/new_method_report.ipynb`]_
_[TODO: `report/fig/images_for_report/Lowland broadleaf.gif` and `report/fig/images_for_report/Highland evergreen.gif`]_
_Results are generated with `_tmp_animate_contunous` or `tmp_animate_contunous_2.py` or `tmp_create_gif.py`._ -->

<a id="T5"></a>
## 5. Implementation details

Most relevant implementation details are reported below. Further details are available in the appendices.

<a id="T5.1"></a>
### 5.1 Continuous processing

_[TODO: add text regarding implementation details. Make a careful selection of what is really needed from sections 5 to 11 of old report `report/new_method_report.ipynb`]_

<a id="T5.2"></a>
### 5.2 COG GeoTIFF generation
_[TODO: 

discuss frequency of generation. 

Which L1/L2/(mask values) are included. 

Discuss what remains excluded (only non-forested area? any other pixels?).

TODO: implement below:
_[E.g. whenever a threshold is exceeded of a ratio of pixels that are at least L2, 
files `YYYYMMDD_historic.tiff` and `YYYYMMDD_historic_mask.tiff` are generated. (TODO: mark previous files with L1 or L0.)]_

]_ 


<a id="T5.3"></a>
### 5.3 In-/output files and their data format 

__COG GeoTIFF__ (`YYYYMMDD_historic.tiff` and `YYYYMMDD_historic_mask.tiff`):

This is the primary output of the historical and continuous processing pipeline.
For each day there exists a file with the processed NDVI values and one with the `mask_array` values (see Table 3 and section 5.2).
_[TODO: mask levels, or L0,L1,L2. E.g. whenever a threshold is exceeded of a ratio of pixels that are at least L2, 
files `YYYYMMDD_historic.tiff` and `YYYYMMDD_historic_mask.tiff` are generated. (TODO: mark previous files with L1 or L0.)]_


| Property | Value |
| :--- | :--- |
| **Driver** | `COG` (Cloud-Optimised GeoTIFF) |
| **Compression** | `DEFLATE` |
| **Data type** | `int16` |
| **CRS** | `EPSG:2056` (CH1903+ / LV95) |
| **Pixel size** | 10 m x 10 m |
| **Extent** | Compact window covering all forest pixels (not full CH bbox) |
| **Naming convention (NDVI)** | `YYYYMMDD.tiff` or `YYYYMMDD_historic.tiff` |
| **Naming convention (mask)** | `YYYYMMDD_mask.tiff` or `YYYYMMDD_historic_mask.tiff` |
| **NO_DATA value** | `NaN` (float stored as `int16` — often rendered as `0` in viewers; use `rasterio` to read correctly) |



__Historical NDVI Zarr__ (`ndvi_historic_vN.zarr`):

This is a secondary (internally used) output of the historical and continuous processing pipeline. 
It is needed as the main input and internal storage structure of the continuous processing pipeline.
It is a Zarr v3 store with a structure defined in Table 3.

Table 2: Definition of fields of the Zarr v3 storage `ndvi_historic_vN.zarr`.
| Field | Description |
| :--- | :--- |
| **Dimensions** | `pixel` (N ≈ 105,715,396 for full CH), `date` (3200+ daily dates from 2017-04-03) |
| **Coordinate: pixel** | `int32`. Pixel ID from 0 to N-1, mapping to forest pixel positions in the reference grid. |
| **Coordinate: date** | `datetime64[ns]`. Daily dates (no gaps). |
| **Coordinate: doy** | (`date`). `int32`. Day of year (1–365, leap days mapped to 365). |
| **Coordinate: x** | (`pixel`). `int32`. Easting in EPSG:2056 (meters), center of pixel. |
| **Coordinate: y** | (`pixel`). `int32`. Northing in EPSG:2056 (meters), center of pixel. |
| **Coordinate: x_idx** | (`pixel`). `int32`. Row index in the 24,542 x 37,728 reference grid. |
| **Coordinate: y_idx** | (`pixel`). `int32`. Column index in the 24,542 x 37,728 reference grid. |
| **Data variable: ndvi_processed** | (`pixel`, `date`). `int16`. Processed NDVI × 10,000. Range: -10,000 to 10,000. `NO_COVERAGE` = 32767, `INVALID` = -32768. |
| **Data variable: mask_array** | (`pixel`, `date`). `int8`. Processing status (see mask legend below). |


The mask_array variable encodes the processing status of each pixel-date combination with integer values as specified in Table 3.

Table 3: Definition of processing status encoded in variable `mask_array` of the Zarr v3 storage `ndvi_historic_vN.zarr`.
| Integer Value | L-level | Meaning |
| :--- | :--- | :--- |
| **0** | L1 | **Not an observation date, not yet smoothed** (data gap between observations) |
| **1** | L2 | **Not an observation date, smoothed** (linear interpolated between (smoothed) observations) |
| **2** | L1 | **Observation date, not yet smoothed** (raw satellite observation) |
| **3** | L2 | **Observation date, valid, smoothed** (satellite observation used for LOESS smoothing, replaced by smoothed value) |
| **4** | L2 | **Observation date, outlier, smoothed** (satellite observation unused for LOESS smoothing, replaced by smoothed value) |

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  In the continuous update pipeline, only dates with mask_array = 3,4 or 1 should be considered 'finalized' L2 products. Values of 2 and 0 in the most recent dates indicate data that has not yet been smoothed due to the latency window (requires 3 observations before and 3 after).
</div>



__Raw Download NDVI/NDSI Zarr__ (`tmp_*_downloaded_*.zarr`):

This is the output from scripts (`MS1_script_for_historical_NDVI/new_historical_processing/1_download_satellite_images.py` and `demo/test_all_pixels/1_extract_swisstopo_dataset.py`)... and used as input for... _[TODO]_

| Field | Description |
| :--- | :--- |
| **Dimensions** | `datetime` (T observations), `pixel` (N forest pixels) |
| **`datetime`** | `datetime64[ns]`. Sub-daily observation timestamps (UTC). |
| **`ndvi`** | `int16`. Raw NDVI × 10,000. `NO_COVERAGE` or `INVALID` for missing/masked pixels. |
| **`ndsi`** | `int16`. Raw NDSI × 10,000. |
| **`x`, `y`, `x_idx`, `y_idx`** | Pixel coordinates (consistent with historic Zarr). |
| **Attribute: `nodata`** | `32767` (`NO_COVERAGE`) |
| **Attribute: `cloud_shadow`** | `-32768` (`INVALID`) |



__Lookup Table `median` Zarr__ (`lookup_table_median_ndvi_v7.zarr`)"

| Field | Description |
| :--- | :--- |
| **Dimensions** | `pixel` (N forest pixels), `doy` (365 days of year) |
| **`median_ndvi`** | `int16`. Expected median NDVI × 10,000 for each pixel and day-of-year, derived from the double-logistic seasonal model (lower + upper envelope average). |
| **Use** | Joined to the processed dataset by `(pixel, doy)` to compute $\Delta\text{-NDVI} = \text{observed} - \text{expected}$, which drives outlier detection and smoothing. |


<a id="References"></a>
## References

<ol>
  <li id="ref1" value="1">Biegel, S., Brüggemann, D., Grossi, F., Volpi, M., Schindler, K., & Stocker, B. D. (2026). Country-wide, high-resolution monitoring of forest browning with Sentinel-2 (arXiv:2604.02074). arXiv. https://doi.org/10.48550/arXiv.2604.02074
  </li>
  <li id="ref2" value="2">swisstopo (2024), swissEO S2-SR. https://www.swisstopo.admin.ch/en/satelliteimage-swisseo-s2-sr (accessed 2026-03-14)</li> 
</ol>

<a id="Appendix"></a>
## Appendix

<a id="AppA"></a>
### Appendix A: Aerial views of focus sites

For the lowland broadleaf case, we select the municipality of Maggia in Ticino, an area near the Maggia river: [map](https://map.geo.admin.ch/#/map?lang=de&center=2694491.82,1126023.20&z=8&topic=ech&bgLayer=ch.swisstopo.swissimage).

The highland broadleaf area is located south from Pizzo della Bassa in Ticino [map](https://map.geo.admin.ch/#/map?lang=de&center=2692020.28,1121443.47&z=8&topic=ech&bgLayer=ch.swisstopo.swissimage).

The lowland evergreen area is located north of Chur, near the Rhine river [map](https://map.geo.admin.ch/#/map?lang=de&center=2761097.61,1194613.45&z=8&topic=ech&bgLayer=ch.swisstopo.swissimage).

The highland evergreen is located east from Davos [map](https://map.geo.admin.ch/#/map?lang=de&center=2781537.00,1182975.00&z=8&topic=ech&bgLayer=ch.swisstopo.swissimage).

The Fire-affected area is found in the municipality of Bitsch, the same location is present in the following tested case [map](https://map.geo.admin.ch/#/map?lang=de&center=2644029.37,1134128.20&z=8&topic=ech&bgLayer=ch.swisstopo.swissimage).

The nearby fire-affected area is found is Bitsch as before [map](https://map.geo.admin.ch/#/map?lang=de&center=2644328.07,1134342.81Y&z=8&topic=ech&bgLayer=ch.swisstopo.swissimage).

We selected an area north of Schaffausen [map](https://map.geo.admin.ch/#/map?lang=de&center=2690025.48,1287413.03&z=8&topic=ech&bgLayer=ch.swisstopo.swissimage) (as illustrated in Fig. 3 in https://onlinelibrary.wiley.com/doi/10.1111/gcb.15360).

The Vaia-storm affected area can be found north from Airolo [map](https://map.geo.admin.ch/#/map?lang=de&center=2689564.74,1154411.88&z=8&topic=ech&bgLayer=ch.swisstopo.swissimage).


<!-- Below boxes can be interactively uncommented for interactive exploration of the locations of focus sites. Alternatively a web browser can be used with the corresponding links. -->

<!-- # # The highland broadleaf area is located south from Pizzo della Bassa in Ticino

# from IPython.display import IFrame, Video, Image, display
# # area location
# x, y = 2692020.28, 1121443.47

# # Construct the URL with your coordinates as center
# url = f"https://map.geo.admin.ch/#/map?lang=de&center={x},{y}&z=10&topic=ech&layers=ch.swisstopo.zeitreihen@year=1864,f;ch.bfs.gebaeude_wohnungs_register,f;ch.bav.haltestellen-oev,f;ch.swisstopo.swisstlm3d-wanderwege,f;ch.vbs.schiessanzeigen,f;ch.astra.wanderland-sperrungen_umleitungen,f&bgLayer=ch.swisstopo.swissimage"
# # Display map in notebook
# IFrame(url, width=1000, height=600) -->

<a id="AppB"></a>
### Appendix B: How to run the historic and continuous processing
_[TODO: decide what to put here and what to put in section 5]_

<a id="AppC"></a>
### Appendix C: Computing environment (dependencies)

This chapter describes how to set up a working Python environment to run the Swiss Forest NDVI processing pipeline. The workflow has been developed and tested on Ubuntu 24.04 LTS and uses a virtualenv-based Python environment.

This chapter also documents hardware, operating system, and system library requirements in detail. It is intended for system administrators and users setting up the pipeline on new machines.


<a id="T1.1"></a>

#### C1.1 Supported Python and Operating System

The pipeline was developed and validated on:

-	Operating System: Ubuntu 24.04 LTS (Noble Numbat), kernel 6.8.0-90-generic
-	Python: 3.10+ (recommended 3.11)
-	Windows: not natively tested; Windows Subsystem for Linux (WSL2 with Ubuntu 22.04 or 24.04) is the recommended path for Windows users
-	macOS: not tested

<a id="T1.2"></a>

#### C1.2 Installation via virtualenv (Recommended)
The primary installation method is a Python virtual environment managed with virtualenv. The setup script 0_0_setup.sh documents all required steps:

``` bash
# 1. Clone the repository
git clone https://github.com/geco-bern/swiss-ndvi-processing.git
cd swiss-ndvi-processing

# 2. Install virtualenv if not already present
sudo apt install python3-virtualenv

# 3. Create the virtual environment
virtualenv .venv

# 4. Activate the environment
source .venv/bin/activate

# 5. Install Python dependencies
pip install -r requirements.txt
``` 
<a id="T1.3"></a>

#### C1.3 Key Python Dependencies
The following packages are central to the pipeline. All versions are pinned in requirements.txt:

| Package | Purpose | Notes |
| :--- | :--- | :--- |
| xarray | Multi-dimensional labelled arrays |	Core data structure throughout |
| zarr (v3)	| Chunked compressed array storage |	Zarr v3 format used |
| dask / dask.distributed |	Parallel and out-of-core computation	| LocalCluster for parallelism |
| rasterio |	Raster I/O and coordinate transforms |	Reads Swisstopo GeoTIFFs |
| rioxarray |	Rasterio extension for xarray |	CRS and COG output |
| pystac_client |	STAC API client for Swisstopo catalog |	Queries ch.swisstopo.swisseo_s2-sr_v100 |
| statsmodels |	LOESS smoothing	| sm.nonparametric.lowess |
| numpy |	Numerical array operations |	Core dependency |
| pandas |	Tabular data and date handling |	DatetimeIndex, date_range |
| numcodecs / zarr.codecs	| Compression codecs |	Blosc/zstd used throughout |
| affine |	Affine coordinate transformations	| Pixel-to-coordinate mapping |
| tqdm	| Progress bars for download loops |	Loop progress in script 1 |

<a id="T1.4"></a>

#### C1.4 System-Level Dependencies
The following system libraries must be present before running pip install:
-	GDAL (>= 3.4): required by rasterio. Install with: sudo apt install gdal-bin libgdal-dev
-	PROJ (>= 8.0): coordinate reference system transformations. Install with: sudo apt install libproj-dev
-	libspatialindex: spatial indexing. Install with: sudo apt install libspatialindex-dev
-	Build tools: sudo apt install build-essential python3-dev

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  If pip install fails for rasterio or fiona, ensure GDAL development headers are installed and that the GDAL version matches the rasterio version pinned in requirements.txt. Use: gdal-config --version to verify.
</div>

<a id="T1.5"></a>

#### C1.5 Alternative: pip + venv (no virtualenv)

``` bash

python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
``` 

<a id="T1.6"></a>

#### C1.6 Activating the Environment
All scripts assume the environment is activated before execution. The pipeline shell scripts activate it automatically via:

``` bash

VENV_PATH="/home/Shared/UniBe-swiss-ndvi/GitHub/swiss-ndvi-processing/.venv"
source "$VENV_PATH/bin/activate"
``` 

For interactive work, activate manually before launching Python or Jupyter:

``` bash

source /path/to/swiss-ndvi-processing/.venv/bin/activate
``` 


#### C2.1 Supported Operating Systems


| OS | Support Level | Notes |
| :--- | :--- | :--- |
| **Ubuntu 24.04 LTS** | Primary / Tested | Main development and production environment. |
| **Ubuntu 22.04 LTS** | Supported | Minor path differences possible. |
| **Other Debian/Ubuntu** | Likely works | Install equivalent system packages. |
| **WSL2 (Ubuntu 22/24)** | Recommended for Windows | Use WSL2, not WSL1; mount data drives under `/mnt/`. |
| **macOS** | Not tested | Conda environment may work; GDAL install differs. |
| **Windows (native)** | Not supported | Use WSL2 instead. |

<a id="T2.2"></a>

#### C2.2 Required System Libraries
Install all system dependencies in a single command on Ubuntu/Debian:

``` bash

sudo apt update && sudo apt install -y \
  gdal-bin libgdal-dev \
  libproj-dev proj-data proj-bin \
  libspatialindex-dev \
  build-essential python3-dev python3-virtualenv \
  libhdf5-dev libnetcdf-dev
``` 

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  libhdf5-dev and libnetcdf-dev are optional but recommended if h5netcdf or scipy-based NetCDF backends are needed.
</div>

<a id="T2.3"></a>

#### C2.3 File Descriptor Limits

The pipeline opens many Zarr chunk files simultaneously. The default OS limit of 1024 open file descriptors is insufficient for large datasets. Increase it before running:

``` bash

# Check current limit
ulimit -n

# Raise for the current session (historic analysis script does this automatically)
ulimit -n 8192
``` 

The shell script 0_1_run_historic_analysis.sh sets this automatically. For production deployments, set permanently in /etc/security/limits.conf:

``` bash

*  soft  nofile  8192
*  hard  nofile  65536
``` 

<a id="T2.4"></a>

#### C2.4 Hardware Requirements
The hardware requirements vary significantly depending on the spatial extent of the dataset (number of pixels) being processed. The codebase was developed and benchmarked on a multi-core workstation (referred to as 'tunder').

| Component | Minimum (demo, ~4K px) | Recommended (full CH, ~105M px) |
| :--- | :--- | :--- |
| **CPU cores** | 4 | 60–120 (Dask workers) |
| **RAM** | 32 GB | 500 GB – 1.5 TB (120 GB/worker x 10) |
| **Disk (data)** | 5 GB | 600 GB – 1.5 TB (Zarr stores + TIFFs) |
| **Disk (temp)** | 2 GB | 400 GB (intermediate Zarr during processing) |
| **Network** | 10 Mbit/s | 100 Mbit/s+ (Swisstopo download ~several GB/year) |



<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  The Dask dashboard (accessible at localhost:8343 during script 4 and 5 execution) provides real-time monitoring of memory and CPU usage. Adjust N_WORKERS and MEMORY_PER_WORKER in the scripts if your workstation has different resources.
</div>

<a id="T2.5"></a>

#### C2.5 Storage Layout

| Path | Purpose |
| :--- | :--- |
| `/mnt/data1/UniBe-swiss-ndvi/` | Primary data store (input/output data, TIFFs) |
| `/mnt/data2/UniBe-swiss-ndvi/` | Scratch space for temporary Zarr files and Dask spill |
| `/home/Shared/UniBe-swiss-ndvi/GitHub/` | Code repository and virtual environment |
| `/mnt/data1/UniBe-swiss-ndvi/backup/` | Backup copies of historical NDVI Zarr |


<a id="AppD"></a>
### Appendix D: Data location, Paths, and Access

This chapter documents all data sources, file paths, and access methods required to run the pipeline. All hard-coded paths in the scripts follow a consistent naming convention.

#### D.1 External Data Source: Swisstopo STAC API

Satellite imagery is downloaded at runtime from the Swisstopo STAC API. No API key or authentication is required — the API is publicly accessible:

-	API endpoint: https://data.geo.admin.ch/api/stac/v0.9/
-	Collection: ch.swisstopo.swisseo_s2-sr_v100 (Sentinel-2 surface reflectance, 10m resolution)
-	Spatial coverage: Switzerland bounding box in WGS84 (lon: 5.70–10.60, lat: 45.80–47.95)
-	Temporal coverage: April 2017 to present
-	Overpass frequency: approximately every 3–5 days depending on orbit

The API returns asset links to cloud-hosted GeoTIFFs (bands-10m.tif, bands-20m.tif, masks-10m.tif). These are read directly with rasterio without downloading the full files.

#### D.2 Reference Grid and Forest Mask

A fixed spatial reference grid defines which pixels are processed. This was determined once from the Swisstopo VHI forest mask and is now hardcoded and stored as a compressed Zarr file:

| Parameter | Value |
| :--- | :--- |
| **Coordinate Reference System** | `EPSG:2056` (CH1903+ / LV95) |
| **Bounding box (L, B, R, T)** | `2474090`, `1065110`, `2851370`, `1310530` [m] |
| **Pixel resolution** | 10 m x 10 m |
| **Full grid size** | 24,542 rows x 37,728 cols = 925,920,576 pixels |
| **Forest pixels** | 105,715,396 pixels (IDs 0 to 105,715,395) |
| **Forest mask file** | `workflow_implementation/data/forest_mask_bits.zarr` |

The forest mask file must be present at the path referenced in script 1. It defines the pixel ID mapping used by all subsequent scripts.

#### D.3 Key Input Data Files

| File / Path | Description | Used by Scripts |
| :--- | :--- | :--- |
| `ndvi_historic_vN.zarr` | Historical processed NDVI time series (2017–present). The main evolving dataset. | 4, 5, 6 |
| `lookup_table_median_ndvi_v7.zarr` | Pre-computed median NDVI per pixel and day-of-year from the double-logistic seasonal model. | 5, 2_historical |
| `forest_mask_bits.zarr` | Packed-bit forest mask defining the 105M pixel domain. | 1 |
| `tmp_*_ndvi_01_downloaded_*.zarr` | **Temporary:** raw downloaded NDVI/NDSI per observation date. | Created by 1, read by 4 |
| `tmp_*_processed.zarr` | **Temporary:** daily resampled and merged NDVI ready for analysis. | Created by 4, read by 5 |

#### D.4 Output Data Files

| File / Path | Description |
| :--- | :--- |
| `data/tiffs/YYYYMMDD.tiff` | Cloud-optimised GeoTIFF of processed NDVI for a given date (COG, int16, EPSG:2056, deflate) |
| `data/tiffs/YYYYMMDD_mask.tiff` | Cloud-optimised GeoTIFF of the mask_array for the same date |
| `data/tiffs_historic_vN/` | Historic TIFF archive — one NDVI + mask TIFF pair per date |
| `ndvi_historic_vN.zarr` (updated) | The historical Zarr is updated in-place (appended) by script 5 |

#### D.5 Temporary Files and Cleanup
The pipeline generates several temporary Zarr stores during processing. These follow the naming convention:


``` bash
tmp_YYYY-MM-DD_HHhMM_ndvi_01_downloaded_<start>_<end>.zarr   # raw download
tmp_YYYY-MM-DD_HHhMM_ndvi_01_downloaded_<start>_<end>A.zarr  # intermediate step
tmp_*_downloaded_*_processed.zarr                             
``` 

These temporary files are not automatically deleted (cleanup lines are commented out in 0_1_run_pipeline.sh). They can be deleted manually after verifying the pipeline completed successfully:

``` bash

rm -rf /mnt/data2/UniBe-swiss-ndvi/data/tmp_*.zarr
``` 

#### D.6 Configuring Paths for a New Machine

All hard-coded paths are concentrated in the shell scripts (0_1_run_pipeline.sh, 0_1_run_historic_analysis.sh) and at the top of each Python script. When deploying on a new machine:
- 1.	Update VENV_PATH to the new virtual environment location
- 2.	Update HISTO_INPUT to the path of the historical NDVI Zarr
- 3.	Update OUTPUT_ZARR / OUTPUT_TIFF_BASE to the desired output directories
- 4.	Update the forest_mask_bits.zarr path in script 1
- 5.	Update DASK_TEMP_DIR to a path with sufficient scratch space

<a id="AppE"></a>
### Appendix E: Hardware and Storage Requirements

This chapter provides concrete storage estimates, RAM benchmarks, and guidance for scaling the pipeline across different spatial extents.

#### E.1 Dataset Size Estimates

The following estimates are based on the current production dataset (Zarr v3, zstd compression, blosc shuffle):

| Dataset | Pixels | Dates | Approx. Size |
| :--- | :--- | :--- | :--- |
| **Demo (10 km x 10 km patch)** | 4,216 | ~3,200 | 40 MB |
| **Small test (1,000 km²)** | 586,503 | ~3,200 | ~5 GB |
| **Regional (10,000 km²)** | 16,041,205 | ~3,200 | ~40 GB |
| **Full Switzerland (forest)** | 105,715,396 | ~3,200 | ~380 GB |
| **Annual TIFF archive (full CH)** | — | ~70 dates/year | ~50 GB/year |

#### E.2 RAM Requirements by Processing Stage

| Script / Stage | RAM Guidance |
| :--- | :--- |
| **Script 1: Download** | **Low (< 8 GB).** Downloads are processed tile-by-tile and written to Zarr incrementally. |
| **Script 4: Merge** (`4_merge_zarr.py`) | **50 workers x 24 GB = 1.2 TB** for full CH. Reduce `N_WORKERS` for smaller machines. |
| **Script 5: Analysis** (`5_analyse_demo_efficient.py`) | **30 workers x 120 GB = 3.6 TB** for full CH. `PIXEL_CHUNKS=40000` controls memory per worker. |
| **Script 2: Historic processing** (`2_historical_ndvi_test.py`) | **80 workers x 20 GB = 1.6 TB.** Batch processing of 200k pixels keeps peak memory manageable. |
| **Script 6: COG TIFF generation** | **Low (< 16 GB).** Grid reconstruction done one date at a time. |

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  The PIXEL_CHUNKS parameter in scripts 4 and 5 is the primary lever for controlling memory use per Dask worker. For a machine with 32 GB RAM and 8 workers, use PIXEL_CHUNKS=5000 and N_WORKERS=8.
</div>

#### E.3 Runtime Benchmarks

Observed runtimes from the production server (60–120 Dask workers):

| Stage | Pixel Count | Observed Runtime |
| :--- | :--- | :--- |
| **Script 1: Download (1 year)** | 105M | ~2–4 hours (network bound) |
| **Script 4: Merge** | 586K | 90 seconds |
| **Script 4: Merge** | 16M | ~15 minutes |
| **Script 5: Analysis** | 586K | 57 seconds |
| **Script 5: Analysis** | 16M | ~55 minutes |
| **Script 2: Historic (full CH, 8yr)** | 105M | ~6 hours per 200K batch |
| **Script 6: TIFF generation (one date)** | 105M | ~5 minutes |